In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from utils.dicece_loss import DiceCELoss
from models.cbam_nnunet import nnUNet3D_CBAM
from utils.boundary_loss import BoundaryLoss
from utils.unets_helper_functions import (
    set_seed,
    train_one_epoch_cbam,
    validate_one_epoch_cbam,
    save_checkpoint,
    PatchDataset_cbam,
    evaluate_full_volume
)

In [ ]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


In [ ]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 12
FOREGROUND_PROB = 0.65
NUM_WORKERS = 2
train_dataset = PatchDataset_cbam(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=True)

val_dataset = PatchDataset_cbam(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=NUM_WORKERS,    
    pin_memory=True
)

In [ ]:
model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("cbam nnU-Net style model Initialized")


In [ ]:
class FinalLoss(nn.Module):
    def __init__(self, weight):
        super().__init__()
        self.dice_ce = DiceCELoss(weight=weight)
        self.boundary = BoundaryLoss()

    def forward(self, logits, targets):
        return self.dice_ce(logits, targets) + 0.3 * self.boundary(logits, targets)

In [ ]:
EPOCHS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

weights = torch.tensor([0.05, 1, 1, 1.2, 1.8, 1.8, 1]).to(device)
loss_fn = FinalLoss(weight=weights)
scaler = torch.cuda.amp.GradScaler()


In [ ]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "cbam_nnunet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float("inf")

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch_cbam(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    val_loss, val_dice = validate_one_epoch_cbam(
        model,
        val_loader,
        loss_fn,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")
    
    scheduler.step()

In [ ]:
# #  RESUME TRAINING 
# PROJECT_ROOT = os.path.abspath("..")
# SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "cbam_nnunet_fold0")

# start_epoch = 0
# best_val_loss = float("inf")

# resume_path = os.path.join(SAVE_DIR, "best_model.pth")  

# if os.path.exists(resume_path):
#     print("Loading checkpoint:", resume_path)

#     checkpoint = torch.load(resume_path, map_location=device)

#     model.load_state_dict(checkpoint["model_state_dict"])
#     optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

#     start_epoch = checkpoint["epoch"] + 1
#     best_val_loss = checkpoint["best_val_loss"]
#     scheduler.last_epoch = start_epoch - 1

#     print(f"Resuming from epoch {start_epoch}")
# else:
#     print("No checkpoint found, starting fresh.")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D_CBAM(
    in_channels=1,
    out_channels=7,
    base_filters=24
).to(device)

model_path = "../experiments/cbam_nnunet_fold0/best_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


In [ ]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)